# PROC INBREEDによる変圧器フリートの設計血縁度の定量化

## エグゼクティブサマリー

送配電事業者の変電所用変圧器は、世代を重ねて設計され、新しいモデルはそれぞれ2つの先代設計から派生する。本notebookはこの設計上の系譜を血統(pedigree)として扱い、**PROC INBREED**を用いて近交係数と相加的血縁(共分散)係数を計算し、任意の2資産がどれだけ設計上の系譜を共有しているかを定量化する。

血統は構造が解釈しやすいように構築されている。現行フリートの4モデルのうち2つは**兄弟(シブリング)**関係にある先代設計のペアから派生しており、系譜が集中している一方、残りの2つは互いに独立した系譜を引いている。手続きはこれを正確に再現する。兄弟由来の2モデル(`G3_T1`、`G3_T3`)はそれぞれ近交係数**F = 0.25**を持ち、残る2モデル(`G3_T2`、`G3_T4`)は**F = 0**である。フリート全体の平均近交係数は**0.0417**である。相加的血縁行列を読むと、現行フリートで最も血縁度が低いペアは**`G3_T1`と`G3_T3`(血縁度 = 0)**であり、両者は共通の祖先を持たず、最も強力な冗長化(N-1)ペアとなる。これは`G3_T1`自体が最も近交度の高い設計の一つであるだけに重要である。

## データソース

| データセット | 行数 | 主要変数 | 説明 |
|---------|------|---------------|-------------|
| `DesignLineage` | 12 | `Generation`, `ID`, `Parent1`, `Parent2`, `Sex` | 3つの重複しない設計世代(創業プラットフォーム4件、第2世代派生機種4件、現行フリート機種4件)にまたがる、変電所用変圧器設計の小規模かつ決定的な工学血統。創業機種以外の各設計には、それが派生した2つの異なる先代設計(`Parent1`、`Parent2`)が記録されている。`Sex`は主導設計の役割(M = 機械/コア系統、F = 電気/巻線系統)を示す。血統は無作為ではなく手動で規定されているため、血縁構造はあらかじめ判明しており、手続きの出力と照合できる。 |

# 変圧器フリートにおける設計系譜の共有度の定量化

## 電力会社が「近親交配」を気にする理由

送配電事業者は数百台の変電所用電力用変圧器を運用している。工学コストと認証作業を抑えるため、メーカーは変圧器を毎回ゼロから設計することはほとんどなく、新しいモデルはそれぞれ1つまたは2つの先代モデルからコア形状、巻線トポロジー、絶縁システム、ブッシング設計を**継承**する。複数回の調達サイクルを経ると、これは真の*工学的系譜*、すなわち設計の血統を生み出す。

この共有された系譜は隠れた信頼性リスクである。同一負荷を保護する2台の変圧器(冗長化された**N-1**ペア)が同じ祖先設計に由来する場合、潜在的な設計上の欠陥――共振モード、絶縁劣化機構、ブッシングのフラッシュオーバー経路など――が**両方**のユニットに存在している可能性が高い。単一の根本原因が冗長ペアを同時に機能停止させる*コモンモード故障*につながりかねない。

**PROC INBREED**は、まさにこの種の系譜の共有度を定量化するために作られた。その起源は動植物の育種にあるが、その数学――ライトの相加的血縁再帰式――は分野を問わない。共通祖先を通じて2つの個体がどれだけ設計上の系譜を共有しているかを測定する。遺伝学の用語を資産工学に対応させると次のとおりである。

| INBREEDの概念 | 電力事業での解釈 |
|---|---|
| 個体(Individual) | 変圧器の設計/モデル |
| 親(Sire/Dam) | それが派生した先代設計 |
| 世代(CLASS) | 調達/設計サイクル |
| 近交係数 *F* | ある設計内での自己相似的な系譜の集中度 |
| 共分散/血縁係数 | 2つの設計間で共有される設計上の系譜 |
| 最も血縁度の低いペア | N-1耐障害性に最適な冗長化ペア |

## ステップ1 ― 設計血統の指定

血縁構造が明確になるよう、`DesignLineage`を直接入力する。設計世代は重複しない3つに分かれる。

- **世代1** ― 先代を持たない(親欄が空欄の)4つの創業プラットフォーム設計(`G1_P1`~`G1_P4`)。
- **世代2** ― それぞれ2つの**異なる**世代1プラットフォームから設計された4つの派生設計(`G2_D1`~`G2_D4`)。`G2_D1`と`G2_D2`は*全同胞*(いずれも`G1_P1`×`G1_P2`由来)であり、`G2_D3`と`G2_D4`も全同胞(いずれも`G1_P3`×`G1_P4`由来)である。
- **世代3** ― 現行フリートの4機種(`G3_T1`~`G3_T4`)。`G3_T1`は同胞ペア`G2_D1`×`G2_D2`から、`G3_T3`は同胞ペア`G2_D3`×`G2_D4`から構築されており、この2設計は系譜が集中する。`G3_T2`と`G3_T4`はそれぞれ2つの独立した系統を交差させている。

`ID`、`Parent1`、`Parent2`の各列が血統を保持し、`Sex`はどちらの工学系統が主導したかを記録する。この後の短いDATAステップで、PROC INBREEDが期待するとおり、創業プラットフォームの親欄が空になるようプレースホルダーの`.`を空白に置き換える。

In [1]:
データ DesignLineage;
   長さ ID $12 Parent1 $12 Parent2 $12 Sex $1;
   INFILE カード truncover;
   入力 Generation ID $ Parent1 $ Parent2 $ Sex $;
   カード;
1 G1_P1 . . M
1 G1_P2 . . M
1 G1_P3 . . M
1 G1_P4 . . F
2 G2_D1 G1_P1 G1_P2 M
2 G2_D2 G1_P1 G1_P2 F
2 G2_D3 G1_P3 G1_P4 F
2 G2_D4 G1_P3 G1_P4 M
3 G3_T1 G2_D1 G2_D2 M
3 G3_T2 G2_D1 G2_D3 F
3 G3_T3 G2_D3 G2_D4 F
3 G3_T4 G2_D2 G2_D4 M
;
実行;

/* 創業プラットフォームには先代が存在しないため、プレースホルダーのドットを空白に置き換える */
データ DesignLineage;
   設定 DesignLineage;
   もし Parent1 = '.' なら Parent1 = ' ';
   もし Parent2 = '.' なら Parent2 = ' ';
実行;

表題 '変圧器設計の系統図';
処理 印刷 データ=DesignLineage noobs 見出;
   変数 Generation ID Parent1 Parent2 Sex;
   見出 Generation='世代'
         ID='設計ID'
         Parent1='親1'
         Parent2='親2'
         Sex='系統区分';
実行;

                                                       変圧器設計の系統図                                                        


    世代      設計ID     親1     親2          系統区分
------  --------  -----  -----  ------------
     1  G1_P1                   M
     1  G1_P2                   M
     1  G1_P3                   M
     1  G1_P4                   F
     2  G2_D1     G1_P1  G1_P2  M
     2  G2_D2     G1_P1  G1_P2  F
     2  G2_D3     G1_P3  G1_P4  F
     2  G2_D4     G1_P3  G1_P4  M
     3  G3_T1     G2_D1  G2_D2  M
     3  G3_T2     G2_D1  G2_D3  F
     3  G3_T3     G2_D3  G2_D4  F
     3  G3_T4     G2_D2  G2_D4  M




NOTE: DATA DesignLineage

NOTE: Processing inline DATALINES (12 lines)

NOTE: Read 12 rows from DATALINES.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA DesignLineage


NOTE: Read 12 rows from DesignLineage.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: Option TITLE changed to 変圧器設計の系統図.
NOTE: PROC PRINT data=DesignLineage

NOTE: PROC PRINT completed: 12 observations printed, 5 variables


## ステップ2 ― 設計世代別の近交係数

`CLASS`ステートメントに`Generation`を指定してPROC INBREEDを**複数世代モード**で実行し、世代別の血統サマリーを出力する。`VAR`ステートメントには、個体・第1先代・第2先代という必須順序で3つの血統列を指定する。

- **IND**オプションは各設計の近交係数*F*(自らの系譜がどれだけ集中しているかの尺度)を出力する。近縁な2つの先代から構築された設計は*F*が高くなる。
- **MATRIX**オプションは完全な相加的血縁行列を出力し、ペアごとの系譜を直接読み取れるようにする。
- **AVERAGE**オプションはフリート全体の平均近交係数(設計多様性を1つにまとめた指標)を追加出力する。

0に近い値は独立した系統を意味し、設計の2つの先代がより近縁になるほど*F*は上昇する。

In [2]:
表題 '設計世代別 近交係数';
処理 inbreed データ=DesignLineage ind average MATRIX;
   変数 ID Parent1 Parent2;
   分類 Generation;
実行;

                                                       設計世代別 近交係数                                                       


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
ID                  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficients (Generation 2)
--------------------------------------
ID                  F (Inbreeding)
G2_D1                 0.000000
G2_D2                 0.000000
G2_D3                 0.000000
G2_D4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coe


NOTE: Option TITLE changed to 設計世代別 近交係数.
NOTE: PROC INBREED data=DesignLineage



## ステップ3 ― 下流のリスクスコアリング用に共分散係数を保存

近交係数の代わりに**COVAR**オプションを使うと、同じ関係性が**共分散(相加的血縁)係数**として報告される。これは資産管理チームが冗長化リスクスコアに取り込む形式である。**OUTCOV=**オプションは完全な行列を(`DesignCovar`という)データセットとして取得し、列`Col1`~`Col12`に各設計と他のすべての設計との関係(血統順)を保持する。これは変電所への割当てに直接結合できる状態になっている。

出力データセットを印字し、保存された行列をリスティングと併せて確認できるようにする。

In [3]:
表題 '共分散(血縁)係数';
処理 inbreed データ=DesignLineage covar MATRIX OUTCOV=DesignCovar;
   変数 ID Parent1 Parent2;
   分類 Generation;
実行;

表題 'OUTCOV=により保存されたリスクスコアリング用の血縁行列';
処理 印刷 データ=DesignCovar noobs;
実行;

                                                       共分散(血縁)係数                                                        


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
ID                  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000

Covariance Matrix (Additive Genetic Relationship) (Generation 1)
--------------------------------------------------
ID                     G1_P1    G1_P2    G1_P3    G1_P4
G1_P1                1.0000   0.0000   0.0000   0.0000
G1_P2                0.0000   1.0000   0.0000   0.0000
G1_P3                0.0000   0.0000   1.0000   0.00


NOTE: Option TITLE changed to 共分散(血縁)係数.
NOTE: PROC INBREED data=DesignLineage

NOTE: OUTCOV= dataset DesignCovar written.
NOTE: Option TITLE changed to OUTCOV=により保存されたリスクスコアリング用の血縁行列.
NOTE: PROC PRINT data=DesignCovar

NOTE: PROC PRINT completed: 12 observations printed, 17 variables


## ステップ4 ― 冗長化(N-1)設置のための最も血縁度の低いペアリング

保存された`DesignCovar`行列は、冗長化検討にまさに必要なものである。設計*i*と設計*j*の関係は行*i*・列`Col`*j*に格納されている(列は血統順なので、`Col9`~`Col12`が現行フリートの4機種`G3_T1`~`G3_T4`に対応する)。

`DesignCovar`から現行フリートの4行を読み込み、すべての順序なしペアを作成し、それぞれに血縁係数を付与して、血縁度が低い順に並べ替える。目的は、共有系譜が**最も低い**冗長化ペアを選ぶことである。そうすることで、1つの継承された設計欠陥がN-1保護位置の両系統を同時に機能停止させる可能性を最小化できる。

In [4]:
/* G3_T1からG3_T4までの関係を持つCol9~Col12の4行を抽出し、
   血統順に並んだ列から各順序なしペアを作成する */
データ Pairs;
   設定 DesignCovar;
   条件 ID IN ('G3_T1','G3_T2','G3_T3','G3_T4');
   長さ DesignA $12 DesignB $12;
   配列 g3{4} Col9 Col10 Col11 Col12;
   配列 nm{4} $12 _temporary_
      ('G3_T1','G3_T2','G3_T3','G3_T4');
   DesignA = ID;
   繰返 j = 1 から 4;
      DesignB = nm{j};
      もし DesignA < DesignB なら 繰返;
         Relationship = g3{j};
         出力;
      終了;
   終了;
   保持 DesignA DesignB Relationship;
実行;

処理 並替 データ=Pairs; 基準 Relationship; 実行;

表題 '冗長化(N-1)候補ペア(血縁度が低い順)';
処理 印刷 データ=Pairs noobs 見出;
   変数 DesignA DesignB Relationship;
   見出 DesignA='設計A'
         DesignB='設計B'
         Relationship='血縁度';
実行;
表題;

                                                 冗長化(N-1)候補ペア(血縁度が低い順)                                                  


    設計A      設計B        血縁度
-------  -------  ---------
G3_T1    G3_T3            0
G3_T2    G3_T4         0.25
G3_T1    G3_T2        0.375
G3_T1    G3_T4        0.375
G3_T2    G3_T3        0.375
G3_T3    G3_T4        0.375




NOTE: DATA Pairs


NOTE: Read 12 rows from DesignCovar.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=Pairs

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 6 rows from Pairs.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: Option TITLE changed to 冗長化(N-1)候補ペア(血縁度が低い順).
NOTE: PROC PRINT data=Pairs

NOTE: PROC PRINT completed: 6 observations printed, 3 variables


## 結果の解釈

**近交係数(ステップ2)。** 世代1の創業プラットフォームと世代2の派生設計はすべて*F* = 0を示す――構造上、いずれも近縁な2つの先代を持たないためである。世代3の2機種で*F* = 0.25が現れる: `G3_T1`(同胞ペア`G2_D1`×`G2_D2`から構築)と`G3_T3`(同胞ペア`G2_D3`×`G2_D4`から構築)である。両者の先代は*同一*の創業プラットフォームのペアに遡るため、系譜が集中している。信頼性の観点からは、これらは単一の継承欠陥に最もさらされやすい設計であり、追加の独立した認証試験が必要である。2つの独立系統を交差させた`G3_T2`と`G3_T4`は*F* = 0である。

**血縁行列(ステップ3)。** 非対角成分はペアごとの系譜共有度を定量化する。世代2の同胞ペアはそれぞれ互いに0.50の血縁度を示し(`G2_D1`-`G2_D2`と`G2_D3`-`G2_D4`)、対する反対系統由来の派生設計は0.00である。近交度の高い現行フリート設計は対角線上に1.25(= 1 + *F*)という自己血縁度を示す。`DesignCovar`データセットにより、完全な行列を変電所割当てと結合できる。

**最も血縁度の低いペアリング(ステップ4)。** すべての現行フリートペアを血縁度で順位付けすると、**`G3_T1`と`G3_T3`が血縁度0.00で第1位**となる――両者は完全に独立した祖先プラットフォームに由来し、設計上の系譜を共有しない。これは最も強力な冗長化ペアであり、`G3_T1`自体が最も近交度の高い2設計の一つであることから特に価値が高い。まったく血縁のない相手とペアを組むことで、その集中した系譜をヘッジできる。次点は`G3_T2`と`G3_T4`のペアで0.25、残りのペアは0.375である。フリート全体の平均近交係数**0.0417**(ステップ2のAVERAGEオプションで出力)は、全体としての設計多様性を要約している。調達においては、N-1のポジションには血縁度が最も低いペアを優先し、時間をかけて祖先基盤を広げるべきである――これは資産工学における遺伝的ボトルネック回避に相当する。

**留意事項。** これは例示用の合成データである。実運用の検討では、メーカーの実際の設計改訂記録から血統を構築し、調達判断を下す前に過去のコモンモード停止事例と血縁度スコアを照合すべきである。